In [1]:
# rsync -av --progress --exclude "*bz2" --exclude "*processed*" login.nird.sigma2.no:/projects/NS9600K/data/islas2022/MRR_islas  /uio/kant/geo-metos-u1/franzihe/Documents/input
# rsync -av --progress login.nird.sigma2.no:/projects/NS9600K/data/islas/alomar_mrr_raw /scratch/franzihe/input/
# bunzip2 -c < 0317.raw.bz2 | gzip -c > 0317.raw.gz


In [2]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)

abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]

if "mimi" in hostname:
    DATA_DIR = "/scratch/franzihe/"
    # FIG_DIR = "/uio/kant/geo-metos-u1/franzihe/Documents/Figures/mrr_reflectivity/"

elif "augustiner" in hostname: 
    DATA_DIR = "/home/franzihe/Data/"
    # FIG_DIR = "/home/franzihe/Documents/Figures/mrr_reflectivity/"

elif "wessel" in hostname:
    DATA_DIR = "/uio/kant/geo-metos-u1/franzihe/Documents/"
    # FIG_DIR = "/uio/kant/geo-metos-u1/franzihe/Documents/Figures/mrr_reflectivity/"


INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'input')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)

# # make figure directory to put in reflectivity processed data
# try:
#     new_dir = pathlib.Path(FIG_DIR)
#     new_dir.mkdir(parents=True, exist_ok=True)
# except OSError:
#     pass




wessel.uio.no


In [3]:
# supress warnings
import warnings
warnings.filterwarnings('ignore')

# import packages
from imports import(glob, datetime, IMProToo, gzip)


In [4]:
version = IMProToo.__version__

In [6]:
# make output directory to put in maahn processed data
try:
    new_dir = pathlib.Path(os.path.join(INPUT_DATA_DIR, 'MRR_islas/maahn'))
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

In [7]:
site = "ANX" #"ALOMAR" #
skipExisting = True

if site == "ALOMAR":
    file_dir = sorted(glob(os.path.join(INPUT_DATA_DIR, 'alomar_mrr_raw/*/*')))
elif site == "ANX":
    file_dir = sorted(glob(os.path.join(INPUT_DATA_DIR, 'MRR_islas/*raw*/*/*')))

In [8]:
# for nfile in :
for nfile in file_dir:
    # get the timestamp
    if nfile.split('.')[-1] == 'gz':
        f = gzip.open(nfile, 'rt')
    else:
        f = open(nfile, 'rb')

# nfile = sorted(glob(os.path.join(INPUT_DATA_DIR, 'MRR_islas/*raw*/*/*')))[1]
# f = gzip.open(nfile, 'rt')

    # Sometimes the first MRR timestamps are from the day before, so we cannot take the first date we found. get list of line breaks
    line_offset = []
    offset = 0
    for line in f:
        line_offset.append(offset)
        offset += len(line)
    f.seek(0)
    # Now, to skip 20% of the file
    f.seek(line_offset[len(line_offset)//5])

    # find the date
    try:
        while True:
            string = str(f.readline())
            string
            
            if not string:
                print('nothing')
                break

            if string[:2] == "T:":
                timestamp = datetime.datetime.strptime(string[2:14], "%y%m%d%H%M%S").strftime("%Y%m%d")
                break
            elif string[:4] == "MRR ":
                timestamp = datetime.datetime.strptime(string[4:16], "%y%m%d%H%M%S").strftime("%Y%m%d")
                break

    finally:
        f.close()


    if timestamp is None:
        print("did not find MRR timestamp in %s, Skipping" %nfile)
        continue

    file_out = "{}_{}_{}_{}.nc".format(os.path.join(OUTPUT_DATA_DIR, 'MRR_islas/maahn/mrr_improtoo'), version, site, timestamp)

    if skipExisting and (os.path.isfile(file_out) or os.path.isfile(file_out+".gz")):
        print("NetCDF file aready exists, skipping: ", timestamp, nfile, file_out)
        continue
    print(timestamp, nfile, file_out)
    
    # load raw data from file
    print("reading...", nfile)
    try:
        rawData = IMProToo.mrrRawData(nfile)
    except:
        print("could not read data")
        continue

    try:
        # convert rawData object
        processedSpec = IMProToo.MrrZe(rawData)
        # average rawData to 60s
        processedSpec.averageSpectra(60)

        # creator attribute of netCDF file
        # all settings (e.g. creator attribute of netCDF file, dealiasing) are available in the 'processedSpec.co' dictionary 
        # and must be set before calculating Ze etc. See the source code for a description of the settings.
        processedSpec.co["ncCreator"] = "F. Hellmuth"
        processedSpec.co["ncDescription"] = "MRR data processed with IMProToo"
        processedSpec.co["ncLocation"] = "Andenes"
        processedSpec.co["ncInstitution"] = "Dept. of Geoscience, University of Oslo"

        # calculate Ze and other moments
        processedSpec.rawToSnow()

        # write all variables to netCDF file.
        print("writing...", file_out)
        processedSpec.writeNetCDF(file_out, ncForm="NETCDF4_CLASSIC")
    except Exception as error:
        print(str(error))
        print("could not process data")
        continue

20210316 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0316.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210316.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0316.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0316.raw.gz
writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210316.nc
20210317 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0317.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210317.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0317.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0317.raw.gz
writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210317.nc
20210318 /uio/kant/geo-metos-u1/franzihe/Documents/i

/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.7/site-packages/IMProToo/core.py:1058: UserWarning: negative (linear) Ze occured during dealiasing, peak removed at timestep 820, bin number 140, most likely at height 2
  t)+', bin number ' + str(ii)+', most likely at height ' + str(peakTmpHeight))
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.7/site-packages/IMProToo/core.py:1042: UserWarning: Truncated peak early. Masked area has wrapped around spectrum width at timestemp 822, bin number 468
  'timestemp ' + str(t) + ', bin number ' + str(ii))


writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210319.nc
20210320 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0320.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210320.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0320.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0320.raw.gz
writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210320.nc
20210321 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0321.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210321.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0321.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0321.raw.gz
writing... /uio/kant/geo-metos-u1/franzihe/Documents

/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.7/site-packages/IMProToo/core.py:1058: UserWarning: negative (linear) Ze occured during dealiasing, peak removed at timestep 745, bin number 1917, most likely at height 29
  t)+', bin number ' + str(ii)+', most likely at height ' + str(peakTmpHeight))
/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.7/site-packages/IMProToo/core.py:1042: UserWarning: Truncated peak early. Masked area has wrapped around spectrum width at timestemp 1433, bin number 398
  'timestemp ' + str(t) + ', bin number ' + str(ii))


writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210322.nc
20210323 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0323.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20210323.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0323.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2021/0323.raw.gz


...

/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.7/site-packages/IMProToo/core.py:1058: UserWarning: negative (linear) Ze occured during dealiasing, peak removed at timestep 776, bin number 1854, most likely at height 28
  t)+', bin number ' + str(ii)+', most likely at height ' + str(peakTmpHeight))


writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20220401.nc
20220402 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0402.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20220402.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0402.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0402.raw.gz


/uio/kant/geo-metos-u1/franzihe/miniconda3/envs/retrieval/lib/python3.7/site-packages/IMProToo/core.py:1058: UserWarning: negative (linear) Ze occured during dealiasing, peak removed at timestep 1030, bin number 1917, most likely at height 29
  t)+', bin number ' + str(ii)+', most likely at height ' + str(peakTmpHeight))


writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20220402.nc
20220403 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0403.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20220403.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0403.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0403.raw.gz
writing... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20220403.nc
20220404 /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0404.raw.gz /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/maahn/mrr_improtoo_0.107_ANX_20220404.nc
reading... /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0404.raw.gz
1 of 1: /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw/2022/0404.raw.gz
writing... /uio/kant/geo-metos-u1/franzihe/Documents

IndexError: list index out of range

In [ ]:
# rsync -av --progress /uio/kant/geo-metos-u1/franzihe/Documents/input/MRR_islas/raw login.nird.sigma2.no:/projects/NS9600K/data/islas2022/MRR_islas